In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import umap
from tqdm import tqdm
import concurrent.futures
import time
import gc
import random
from typing import List, Tuple, Optional
import threading

# Datashader and visualization libraries
import datashader as ds
import datashader.transfer_functions as tf
import holoviews as hv
import holoviews.operation.datashader as hd

# Enable HoloViews with Bokeh backend for interactive plots
hv.extension('bokeh')

print("✓ All libraries imported successfully!")


In [ ]:
# Configure paths
places_dir = '/data0/teja_codes/ImmersoAiResearch/ImageMatchingAlgo/image_search/place_search'
frames_dir = '/data0/teja_dataset_embeddings/clip_embeddings'

# Configure processing parameters
MAX_WORKERS = min(32, (os.cpu_count() or 1) + 4)
BATCH_SIZE = 2000
MAX_FRAME_FILES = 50000  # Limit frame files for memory management
EXPECTED_DIM = 768  # CLIP embedding dimension

# Set numpy threading for performance
os.environ['OMP_NUM_THREADS'] = str(int(os.cpu_count() * 0.25))
os.environ['MKL_NUM_THREADS'] = str(int(os.cpu_count() * 0.25))

print(f"Configuration:")
print(f"  - Max workers: {MAX_WORKERS}")
print(f"  - Batch size: {BATCH_SIZE:,}")
print(f"  - Max frame files: {MAX_FRAME_FILES:,}")
print(f"  - Expected embedding dimension: {EXPECTED_DIM}")


In [ ]:
class EmbeddingLoader:
    def __init__(self, max_workers: Optional[int] = None, batch_size: int = 1000):
        self.max_workers = max_workers or MAX_WORKERS
        self.batch_size = batch_size
        self.lock = threading.Lock()
        
    def collect_embeddings(self, root_dir: str, label_tag: str) -> Tuple[List[str], List[str]]:
        """Collect all .npy file paths with their labels"""
        paths, labels = [], []
        
        # Files to skip (common non-embedding files)
        skip_files = {'final_labels.npy', 'labels.npy', 'metadata.npy', 'index.npy'}
        
        print(f"Scanning directory: {root_dir}")
        for dirpath, _, filenames in tqdm(os.walk(root_dir), desc=f"Scanning {label_tag} dirs"):
            npy_files = [f for f in filenames if f.endswith('.npy') and f not in skip_files]
            for fname in npy_files:
                paths.append(os.path.join(dirpath, fname))
                labels.append(label_tag)
        
        return paths, labels

    def load_single_embedding(self, path_label: Tuple[str, str]) -> Tuple[np.ndarray, List[str]]:
        """Load a single embedding file with error handling"""
        path, label = path_label
        try:
            emb = np.load(path)
            
            if emb.ndim == 1:
                if emb.shape[0] != EXPECTED_DIM:
                    return np.array([]), []
                return emb.reshape(1, -1), [label]
            elif emb.ndim == 2:
                if emb.shape[1] != EXPECTED_DIM:
                    return np.array([]), []
                return emb, [label] * emb.shape[0]
            else:
                return np.array([]), []
        except Exception as e:
            return np.array([]), []

    def load_embeddings_batch(self, path_labels: List[Tuple[str, str]]) -> Tuple[List[np.ndarray], List[str], int]:
        """Load a batch of embeddings in parallel"""
        batch_embeddings = []
        batch_labels = []
        files_skipped = 0
        
        with concurrent.futures.ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = {executor.submit(self.load_single_embedding, pl): pl for pl in path_labels}
            
            for future in tqdm(concurrent.futures.as_completed(futures), 
                             total=len(futures), 
                             desc="Loading batch",
                             leave=False):
                try:
                    embs, labels = future.result()
                    if embs.size > 0:
                        batch_embeddings.append(embs)
                        batch_labels.extend(labels)
                    else:
                        files_skipped += 1
                except Exception as e:
                    files_skipped += 1
        
        return batch_embeddings, batch_labels, files_skipped

    def load_all_embeddings(self, all_paths: List[str], all_labels: List[str]) -> Tuple[np.ndarray, List[str]]:
        """Load all embeddings in batches with memory management"""
        total_files = len(all_paths)
        all_embeddings = []
        final_labels = []
        total_skipped = 0
        
        print(f"Loading {total_files:,} embedding files in batches of {self.batch_size:,}")
        
        for i in tqdm(range(0, total_files, self.batch_size), desc="Processing batches"):
            batch_end = min(i + self.batch_size, total_files)
            batch_paths = all_paths[i:batch_end]
            batch_labels = all_labels[i:batch_end]
            
            path_labels = list(zip(batch_paths, batch_labels))
            batch_embeddings, batch_final_labels, batch_skipped = self.load_embeddings_batch(path_labels)
            
            total_skipped += batch_skipped
            all_embeddings.extend(batch_embeddings)
            final_labels.extend(batch_final_labels)
            
            if i > 0 and i % (self.batch_size * 10) == 0:
                gc.collect()
        
        # Convert to numpy array and remove duplicates
        if not all_embeddings:
            raise ValueError("No valid embeddings were loaded!")
        
        valid_embeddings = [emb for emb in all_embeddings if emb.size > 0]
        if not valid_embeddings:
            raise ValueError("No valid embeddings were loaded!")
        
        print(f"\\nConcatenating {len(valid_embeddings):,} valid embedding arrays...")
        final_embeddings = np.concatenate(valid_embeddings, axis=0)
        final_labels_array = np.array(final_labels)
        
        print(f"Removing duplicate embeddings...")
        unique_embeddings, unique_indices = np.unique(final_embeddings, axis=0, return_index=True)
        unique_labels = final_labels_array[unique_indices].tolist()
        
        duplicates_removed = final_embeddings.shape[0] - unique_embeddings.shape[0]
        print(f"Removed {duplicates_removed:,} duplicate embeddings")
        print(f"Final shape: {unique_embeddings.shape}")
        
        return unique_embeddings, unique_labels

print("✓ EmbeddingLoader class defined!")


In [ ]:
# Initialize loader
loader = EmbeddingLoader(max_workers=MAX_WORKERS, batch_size=BATCH_SIZE)

# Collect paths
print("📁 Collecting embedding file paths...")
place_paths, place_labels = loader.collect_embeddings(places_dir, "place")
frame_paths, frame_labels = loader.collect_embeddings(frames_dir, "frame")

# Randomly shuffle frame files and limit to MAX_FRAME_FILES
frames = [(frame_path, frame_label) for frame_path, frame_label in zip(frame_paths, frame_labels)]
random.shuffle(frames)
frame_paths = [frame[0] for frame in frames]
frame_labels = [frame[1] for frame in frames]
frame_paths = frame_paths[:MAX_FRAME_FILES]
frame_labels = frame_labels[:MAX_FRAME_FILES]

# Combine all paths
all_paths = place_paths + frame_paths
all_labels = place_labels + frame_labels

print(f"\\n📊 File collection summary:")
print(f"  - Place files: {len(place_paths):,}")
print(f"  - Frame files: {len(frame_paths):,}")
print(f"  - Total files: {len(all_paths):,}")


In [ ]:
print("🔄 Loading embeddings...")
start_time = time.time()

all_embeddings, final_labels = loader.load_all_embeddings(all_paths, all_labels)

loading_time = time.time() - start_time

print(f"\\n✅ Loading completed!")
print(f"  - Total embeddings: {all_embeddings.shape[0]:,}")
print(f"  - Embedding dimension: {all_embeddings.shape[1]}")
print(f"  - Loading time: {loading_time:.2f} seconds")
print(f"  - Loading rate: {len(all_embeddings)/loading_time:.1f} embeddings/second")

# Check label distribution
labels_df = pd.DataFrame({'label': final_labels})
print(f"\\n📊 Label distribution:")
print(labels_df['label'].value_counts())


In [ ]:
print("🔬 Running PCA...")
pca_start = time.time()

# Apply PCA to reduce from 768 to 50 dimensions
pca = PCA(n_components=50, random_state=42)
pca_result = pca.fit_transform(all_embeddings)

pca_time = time.time() - pca_start

print(f"✅ PCA completed!")
print(f"  - Original shape: {all_embeddings.shape}")
print(f"  - PCA result shape: {pca_result.shape}")
print(f"  - PCA time: {pca_time:.2f} seconds")
print(f"  - Explained variance ratio (first 10 components): {pca.explained_variance_ratio_[:10].round(4)}")
print(f"  - Total explained variance: {pca.explained_variance_ratio_.sum():.4f}")

# Clear memory
del all_embeddings
gc.collect()
print("🗑️ Cleared original embeddings from memory")


In [ ]:
print("🗺️ Running UMAP...")
umap_start = time.time()

# Apply UMAP to reduce from 50 to 2 dimensions
umap_model = umap.UMAP(
    n_components=2, 
    random_state=42,
    n_jobs=-1,  # Use all available cores
    verbose=True,  # Show UMAP progress
    n_neighbors=15,
    min_dist=0.1
)

umap_result = umap_model.fit_transform(pca_result)

umap_time = time.time() - umap_start

print(f"\\n✅ UMAP completed!")
print(f"  - PCA result shape: {pca_result.shape}")
print(f"  - UMAP result shape: {umap_result.shape}")
print(f"  - UMAP time: {umap_time:.2f} seconds")
print(f"  - X range: [{umap_result[:, 0].min():.2f}, {umap_result[:, 0].max():.2f}]")
print(f"  - Y range: [{umap_result[:, 1].min():.2f}, {umap_result[:, 1].max():.2f}]")

# Clear PCA results from memory
del pca_result
gc.collect()
print("🗑️ Cleared PCA results from memory")


In [ ]:
print("📊 Creating visualization DataFrame...")

# Create DataFrame for Datashader
viz_df = pd.DataFrame({
    'x': umap_result[:, 0],
    'y': umap_result[:, 1],
    'category': final_labels
})

print(f"✅ DataFrame created!")
print(f"  - Shape: {viz_df.shape}")
print(f"  - Memory usage: {viz_df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\\n📊 Category distribution:")
print(viz_df['category'].value_counts())

print(f"\\n📊 Data preview:")
print(viz_df.head())
print(viz_df.describe())
